In [1]:
import sys
import os
import re
import pickle
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from thefuzz import process

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.regex_extractor import extract_volume_percent

print("Информация: Все необходимые модули импортированы.")

mock_master_dictionary = {
    "known_brands": sorted(["Сады Придонья", "Простоквашино", "Мираторг", "Сады"], key=len, reverse=True),
    "atomic_types": {"сок", "вода", "пюре", "молоко", "кефир", "котлеты"},
    "stop_words": {"классический", "отборный", "сливочный"}
}
print("Информация: Созданы тестовые 'заглушки' для словарей.")

try:
    tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
    print("Информация: Токенизатор 'xlm-roberta-base' успешно загружен.")
except Exception as e:
    print(f"Ошибка: Не удалось загрузить токенизатор. Убедитесь, что есть интернет-соединение. Ошибка: {e}")
    tokenizer = None

Информация: Все необходимые модули импортированы.
Информация: Созданы тестовые 'заглушки' для словарей.
Информация: Токенизатор 'xlm-roberta-base' успешно загружен.


In [2]:
def tag_product_name(product_item, master_dictionary):
    name = product_item.get("name", "")
    weight = product_item.get("weight", "")
    category_name = product_item.get("category_name", "")

    known_brands = master_dictionary["known_brands"][:]
    atomic_types = master_dictionary["atomic_types"]

    char_tags = ['O'] * len(name)

    def _mark_chars(start, end, label):
        if start is None or end is None:
            return False
        if start < 0 or end > len(name) or start >= end:
            return False

        if any(char_tags[i] != 'O' for i in range(start, end)):
            return False

        char_tags[start] = f'B-{label}'
        for i in range(start + 1, end):
            char_tags[i] = f'I-{label}'
        return True


    if name:
        entities_from_regex_in_name = extract_volume_percent(name)
        for start, end, label in entities_from_regex_in_name:
            _mark_chars(start, end, label.split('-')[1])

    if weight and name:
        entities_from_regex_in_weight = extract_volume_percent(weight)
        for w_start, w_end, w_label in entities_from_regex_in_weight:
            fragment = weight[w_start:w_end]
            num_m = re.search(r"\d[\d\.,]*", fragment)
            if not num_m:
                continue
            number_text = num_m.group(0)
            for m in re.finditer(re.escape(number_text), name):
                n_start, n_end = m.span()
                _mark_chars(n_start, n_end, w_label.split('-')[1])

    for brand in known_brands:
        for match in re.finditer(re.escape(brand), name, re.IGNORECASE):
            _mark_chars(match.start(), match.end(), "BRAND")

    types_from_category = {t.strip() for t in re.split(r',|/| и ', category_name.lower()) if len(t.strip()) > 2}
    relevant_types = sorted(list(atomic_types.intersection(types_from_category)), key=len, reverse=True)
    for type_word in relevant_types:
        for match in re.finditer(r'\b' + re.escape(type_word) + r'\b', name, re.IGNORECASE):
            _mark_chars(match.start(), match.end(), "TYPE")

    unmarked_segments = []
    start_idx = 0
    for i, tag in enumerate(char_tags):
        if tag != 'O' and start_idx is not None:
            if i > start_idx:
                unmarked_segments.append((name[start_idx:i], start_idx))
            start_idx = None
        elif tag == 'O' and start_idx is None:
            start_idx = i
    if start_idx is not None and start_idx < len(name):
        unmarked_segments.append((name[start_idx:], start_idx))

    for segment, offset in unmarked_segments:
        segment_clean = re.sub(r'[\W_]+', '', segment, flags=re.UNICODE)
        if not segment_clean or len(segment_clean) < 2:
            continue

        try:
            extracted = process.extractOne(segment, known_brands)
            best_brand, brand_score = extracted if extracted else (None, 0)
        except Exception:
            best_brand, brand_score = None, 0
        if best_brand and brand_score >= 90:
            match = re.search(re.escape(best_brand), segment, re.IGNORECASE)
            if match:
                _mark_chars(offset + match.start(), offset + match.end(), "BRAND")

        try:
            extracted_t = process.extractOne(segment, list(atomic_types))
            best_type, type_score = extracted_t if extracted_t else (None, 0)
        except Exception:
            best_type, type_score = None, 0
        if best_type and type_score >= 90:
            match = re.search(re.escape(best_type), segment, re.IGNORECASE)
            if match:
                _mark_chars(offset + match.start(), offset + match.end(), "TYPE")

    unmarked_segments = []
    start_idx = 0
    for i, tag in enumerate(char_tags):
        if tag != 'O' and start_idx is not None:
            if i > start_idx:
                unmarked_segments.append((name[start_idx:i], start_idx))
            start_idx = None
        elif tag == 'O' and start_idx is None:
            start_idx = i
    if start_idx is not None and start_idx < len(name):
        unmarked_segments.append((name[start_idx:], start_idx))

    for segment, offset in unmarked_segments:
        segment_clean = re.sub(r'[\W_]+', '', segment, flags=re.UNICODE)
        if not segment_clean or len(segment_clean) < 3:
            continue

        for match in re.finditer(r'\b([A-ZА-Я][a-zA-Zа-яА-Я]{2,}|[a-zA-Z]{3,})\b', segment):
            candidate = match.group(0)

            try:
                extracted_b = process.extractOne(candidate, known_brands)
                best_known_brand, score = extracted_b if extracted_b else (None, 0)
            except Exception:
                best_known_brand, score = None, 0

            if score < 85:
                if _mark_chars(offset + match.start(), offset + match.end(), "BRAND"):
                    known_brands.append(candidate)
                    known_brands.sort(key=len, reverse=True)

    if tokenizer is None:
        tokens = list(name)
        final_bio_tags = char_tags[:len(tokens)]
        return tokens, final_bio_tags

    encoding = tokenizer(name, return_offsets_mapping=True, add_special_tokens=False)
    tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])
    offsets = encoding["offset_mapping"]

    final_bio_tags = []
    for tok_start, tok_end in offsets:
        if 0 <= tok_start < len(char_tags):
            final_bio_tags.append(char_tags[tok_start])
        else:
            final_bio_tags.append('O')

    return tokens, final_bio_tags


print("Информация: Главная функция разметки `tag_product_name` определена.")

Информация: Главная функция разметки `tag_product_name` определена.


In [3]:
test_items = [
    {
        "name": "Сок Сады Придонья яблочный 0,2л",
        "weight": "200 мл",
        "category_name": "Соки, нектары"
    },
    {
        "name": "Молоко Простоквашино паст. 3,2% 930мл",
        "weight": "930 мл",
        "category_name": "Молоко, сливки"
    },
    {
        "name": "Котлеты Мираторг из говядины",
        "weight": "400 г",
        "category_name": "Мясная гастрономия"
    },
    {
        "name": "Вода Святой Источник без газа 5л",
        "weight": "5 л",
        "category_name": "Вода, напитки"
    },
    {
        "name": "Ряженка Братья Чебурашкины 3.2-4.2% БЗМЖ 330г",
        "weight": "330 г",
        "category_name": "Кефир, творог"
    },
    {
        "name": "млоко Простоквашино 1л",
        "weight": "1 л",
        "category_name": "Молоко, сливки"
    },
    {
        "name": "Котлеты Мираторгг домашние",
        "weight": "400 г",
        "category_name": "Мясная гастрономия"
    },
    {
        "name": "Конфеты Марсианка Трио",
        "weight": "200 г",
        "category_name": "Конфеты, шоколад"
    }
]

print("--- Запуск тестирования алгоритма на примерах ---")
for item in test_items:
    tokens, tags = tag_product_name(item, mock_master_dictionary)

    print(f"\nНазвание: {item['name']}")
    print("Результат разметки:")
    for token, tag in zip(tokens, tags):
        print(f"  {token:<15} {tag}")

--- Запуск тестирования алгоритма на примерах ---

Название: Сок Сады Придонья яблочный 0,2л
Результат разметки:
  ▁С              B-TYPE
  ок              I-TYPE
  ▁Са             B-BRAND
  ды              I-BRAND
  ▁При            I-BRAND
  дон             I-BRAND
  ья              I-BRAND
  ▁я              O
  бло             O
  чный            O
  ▁0,2            B-VOLUME
  л               I-VOLUME

Название: Молоко Простоквашино паст. 3,2% 930мл
Результат разметки:
  ▁Мол            B-TYPE
  око             I-TYPE
  ▁Просто         B-BRAND
  ква             I-BRAND
  ши              I-BRAND
  но              I-BRAND
  ▁па             O
  ст              O
  .               O
  ▁3              B-PERCENT
  ,               I-PERCENT
  2%              I-PERCENT
  ▁9              B-VOLUME
  30              I-VOLUME
  м               I-VOLUME
  л               I-VOLUME

Название: Котлеты Мираторг из говядины
Результат разметки:
  ▁Кот            B-TYPE
  лет             I-TYPE
  ы     